# Artificial Neural Network — Customer Churn Prediction

## The Business Problem

A bank has 10,000 customers. Some of them are leaving (churning). The bank wants to predict **which customers are likely to leave** so it can proactively retain them.

Losing a customer costs the bank money — retaining an at-risk customer with an offer is cheaper than acquiring a new one. This is a **binary classification** problem:
- Output `1` = customer will leave
- Output `0` = customer will stay

---

## Why a Neural Network?

This dataset has **12 features** — a mix of numerical (age, salary, balance) and categorical (country, gender) variables. The relationship between these features and churn is complex and non-linear:

- A 55-year-old customer with a large balance in Germany churns at a very different rate than a 25-year-old with a small balance in France
- Simple linear models cannot capture these multi-feature interactions

Neural networks learn **hierarchical combinations of features** through multiple layers, making them powerful for problems where the interactions between variables are complex.

**For this specific dataset, XGBoost or Random Forest might actually achieve similar accuracy** — neural networks are not always the best tool for tabular data. This notebook teaches the architecture and workflow rather than claiming NNs dominate.

---

## What We Will Build

```
Input (12 features)
        ↓
  Dense layer (6 neurons, ReLU)
        ↓
  Dense layer (6 neurons, ReLU)
        ↓
  Output (1 neuron, Sigmoid) → P(churn)
```

A shallow 2-hidden-layer network. We will train it with Adam optimiser and binary cross-entropy loss over 100 epochs.

### Step 1: Import Libraries

| Library | Why we need it |
|---------|---------------|
| `numpy` | Array operations and reshaping for predictions |
| `pandas` | Loading and inspecting the dataset |
| `tensorflow` | Building, training, and running the neural network (Keras API) |

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

In [ ]:
tf.__version__

## Part 1: Data Preprocessing

Neural networks require more preprocessing than tree-based models. We need to:

1. **Encode categorical variables** — the network only works with numbers
2. **Split** — never fit preprocessing on test data
3. **Scale features** — neural networks are particularly sensitive to feature magnitude differences

Each of these steps has a specific reason, explained below.

### Step 2: Load the Dataset

The Churn Modelling dataset contains **10,000 bank customers** with 14 columns:

| Columns we drop | Reason |
|-----------------|--------|
| `RowNumber` (col 0) | Just an index, no information |
| `CustomerId` (col 1) | Arbitrary identifier, not a feature |
| `Surname` (col 2) | Not predictive of churn |

We use `iloc[:, 3:-1]` to take columns 3 through 12 as features, and `iloc[:, -1]` as the target (`Exited`).

**Features used:**

| Feature | Type |
|---------|------|
| CreditScore | Numerical |
| Geography | Categorical (France, Germany, Spain) |
| Gender | Categorical (Male, Female) |
| Age | Numerical |
| Tenure | Numerical |
| Balance | Numerical |
| NumOfProducts | Numerical |
| HasCrCard | Binary |
| IsActiveMember | Binary |
| EstimatedSalary | Numerical |

In [ ]:
dataset = pd.read_csv('Churn_Modelling.csv')
X = dataset.iloc[:, 3:-1].values
y = dataset.iloc[:, -1].values

In [ ]:
print(X)

In [ ]:
print(y)

### Step 3: Encode Categorical Variables

Neural networks compute weighted sums of their inputs: $z = w_1 x_1 + w_2 x_2 + ...$

This only works with numbers. We have two categorical columns:

| Column | Values | Encoding strategy |
|--------|--------|-------------------|
| `Gender` | Male / Female | **Label encoding** (2 categories, binary relationship) |
| `Geography` | France / Germany / Spain | **One-hot encoding** (3 categories, no ordinal relationship) |

The encoding choices matter — using the wrong one introduces false mathematical relationships.

#### Label Encoding: Gender

Label encoding converts text categories to integers: Female → 0, Male → 1.

**Why label encoding is appropriate for Gender:**

There are only 2 values. With two classes, label encoding and one-hot encoding are equivalent — the model can learn that 0 and 1 represent two distinct categories without implying any ordering.

**Why we cannot label-encode Geography:**

If we encoded France=0, Germany=1, Spain=2, the model would treat Spain as mathematically twice Germany. That arithmetic relationship does not exist — these are just three different countries. Label encoding multi-class nominal variables introduces a spurious ordinal relationship.

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X[:, 2] = le.fit_transform(X[:, 2])

In [ ]:
print(X)

#### One-Hot Encoding: Geography

One-hot encoding creates a **separate binary column** for each category:

```
Geography   →   France  Germany  Spain
France           1        0       0
Germany          0        1       0
Spain            0        0       1
```

Each country is now independent — the model can learn a separate weight for each without any implied ordering or arithmetic relationship between them.

`ColumnTransformer` applies the `OneHotEncoder` to column index `[1]` (Geography) and passes through all other columns unchanged (`remainder='passthrough'`). We wrap in `np.array()` because `ColumnTransformer` returns a sparse matrix by default.

**Result:** X now has 12 columns (3 one-hot geography columns + 9 original numerical columns).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [1])], remainder='passthrough')
X = np.array(ct.fit_transform(X))

In [ ]:
print(X)

### Step 4: Train/Test Split

We hold out 20% (2,000 customers) as the final test set.

The split happens **before scaling and before network training** — the test set must never influence any part of the model preparation pipeline. This is the only way to get an unbiased estimate of how the model will perform on new customers.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

### Step 5: Feature Scaling (Critical for Neural Networks)

**This step is not optional for neural networks.** Here is why it matters more than for tree models:

Neural networks update weights via gradient descent. Gradients are computed as partial derivatives of the loss with respect to each weight. If feature scales differ dramatically:

```
EstimatedSalary: 150,000   → gradient for its weight is tiny
IsActiveMember:  1          → gradient for its weight is large
```

The optimiser has to use the same learning rate for all weights. With unscaled features, the learning rate that works for large-scale features will be too large for small-scale features and vice versa — training becomes unstable or converges to a poor solution.

`StandardScaler` centres each feature to mean 0, standard deviation 1. After scaling, all features contribute comparable gradient magnitudes and training is well-conditioned.

**Rule:** Fit on training data only. Transform both sets with the same fitted scaler.

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## Part 2: Building the Neural Network

We will build a **feedforward neural network** using the Keras `Sequential` API. This architecture is a series of fully-connected (Dense) layers:

```
Layer           Neurons    Activation    Purpose
------          -------    ----------    -------
Input           12         -             One neuron per feature
Hidden 1        6          ReLU          Learn first-order feature combinations
Hidden 2        6          ReLU          Learn higher-order combinations
Output          1          Sigmoid       Output probability of churn (0 to 1)
```

The choice of 6 neurons per hidden layer is a rule of thumb: roughly the average of input size (12) and output size (1). In practice, you would tune this with cross-validation.

### Step 6: Initialise the Network

`tf.keras.models.Sequential()` creates a model where layers are stacked one after another — the output of each layer feeds directly into the next.

This is the right architecture for a standard feedforward network. For architectures with branches, skip connections, or multiple inputs/outputs, you would use the Keras Functional API instead.

In [ ]:
ann = tf.keras.models.Sequential()

### Step 7: Add the First Hidden Layer

```python
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))
```

**`Dense`** means every neuron in this layer is connected to every neuron in the previous layer (fully connected).

**`units=6`** — 6 neurons in this layer. Each neuron learns a different linear combination of the 12 input features, then passes it through the activation function.

**`activation='relu'`** — ReLU (Rectified Linear Unit): $f(z) = \max(0, z)$

**Why ReLU?**

Without an activation function, stacking Dense layers gives you nothing more than a single linear transformation — no matter how many layers you add. The activation introduces non-linearity, which is what allows the network to learn complex patterns.

ReLU is the default for hidden layers because:
- Gradient is 1 for positive inputs → no vanishing gradient during backpropagation
- Computationally trivial (just clamp negatives to zero)
- Works well in practice across most architectures

**Note:** Keras automatically infers the input shape from the first batch of training data, so we do not need to specify `input_dim` explicitly.

In [ ]:
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))

### Step 8: Add the Second Hidden Layer

A second hidden layer allows the network to learn **higher-order combinations** of the features learned in the first layer.

Think of it like this:
- Layer 1 might learn: "older + high balance", "young + Germany", "inactive + multiple products"
- Layer 2 combines these: "older + high balance + inactive" → strong churn signal

**How many hidden layers do you need?**

| Layers | Capability |
|--------|------------|
| 0 | Linear model only |
| 1 | Can approximate any continuous function (universal approximation theorem) |
| 2 | More efficient representation of complex functions |
| 3+ | Deep learning — for images, text, sequences; often overkill for tabular data |

Two hidden layers is a common starting point for tabular classification. More layers are not always better — they increase training time and overfit risk.

In [ ]:
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))

### Step 9: Add the Output Layer

```python
ann.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))
```

**`units=1`** — one output neuron because we are doing **binary classification** (churn or no churn).

**`activation='sigmoid'`** — the sigmoid function squashes the output to the range (0, 1):

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

This gives us a **probability**: the model outputs P(customer churns). We then apply a threshold (typically 0.5) to make a binary decision.

**Why sigmoid in the output layer only?**

Sigmoid is not used in hidden layers in modern networks because it saturates near 0 and 1 — the gradient becomes essentially zero, causing the **vanishing gradient problem** during backpropagation. Early layers stop learning. ReLU avoids this in hidden layers, while sigmoid remains the correct choice for the final binary output.

In [ ]:
ann.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

## Part 3: Training the Network

Training a neural network has three phases per epoch:

1. **Forward pass** — feed inputs through all layers, compute the prediction
2. **Loss computation** — measure how wrong the prediction is
3. **Backward pass (backpropagation)** — compute gradients of the loss with respect to every weight, then update weights via the optimiser

This cycle repeats for every mini-batch, and an **epoch** is one complete pass through the entire training set.

### Step 10: Compile the Network

Compiling specifies three things:

**`optimizer='adam'`** — Adam (Adaptive Moment Estimation) is the standard optimiser for deep learning. It:
- Adapts the learning rate individually for each weight based on the history of gradients
- Combines momentum (smooths gradient direction) and RMSprop (adapts learning rate)
- Converges faster than vanilla SGD and is less sensitive to the initial learning rate

**`loss='binary_crossentropy'`** — the standard loss function for binary classification:

$$\mathcal{L} = -[y \log(\hat{y}) + (1-y) \log(1-\hat{y})]$$

It penalises confident wrong predictions heavily. Predicting 0.99 when the true label is 0 results in a much larger loss than predicting 0.6. This is what drives the model to be well-calibrated, not just accurate.

**`metrics=['accuracy']`** — displayed during training for monitoring. Note: accuracy is **not** what the optimiser minimises (it minimises cross-entropy). Accuracy is logged purely for human readability.

In [ ]:
ann.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

### Step 11: Train the Network

**`batch_size=32`** — instead of computing the gradient over all 8,000 training examples before updating weights (slow), we compute gradients on random mini-batches of 32 samples.

Why 32?
- Small enough that each batch is computed quickly
- Large enough that the gradient estimate is not too noisy
- 32 is the most common default in practice; values of 64, 128, 256 are also common

**`epochs=100`** — the training set is passed through the network 100 times.

Each epoch: 8,000 training samples / 32 batch size = 250 weight updates.
Total weight updates: 100 epochs x 250 = 25,000 gradient steps.

**What to watch during training:**

- Loss should decrease each epoch — the model is learning
- If loss plateaus early, try a different learning rate or more neurons
- If loss oscillates wildly, the learning rate is too high
- We are not using a validation split here — in practice, always add `validation_split=0.1` to detect overfitting during training

In [ ]:
ann.fit(X_train, y_train, batch_size = 32, epochs = 100)

## Part 4: Making Predictions and Evaluating the Model

The model is trained. Now we evaluate it on data it has never seen.

Two types of evaluation:
1. **Single observation prediction** — simulates how you would use the model in production for one customer
2. **Full test set evaluation** — overall performance metrics on 2,000 unseen customers

### Step 12: Predict a Single Customer

This exercise demonstrates the complete inference pipeline for a new customer.

**Critical preprocessing steps for a single prediction:**

1. **Encode Geography manually** — France was one-hot encoded as `[1, 0, 0]` (check the order your encoder created)
2. **Encode Gender manually** — Male = 1, Female = 0 (from our LabelEncoder)
3. **Wrap in double brackets** — `predict()` expects a 2D array. A single observation needs shape `(1, 12)`, not `(12,)`
4. **Apply the same scaler** — use `sc.transform()`, not `sc.fit_transform()`. The scaler was fit on training data — you must use those same statistics for every new prediction

**Homework**

Use our ANN model to predict if the customer with the following informations will leave the bank: 

Geography: France

Credit Score: 600

Gender: Male

Age: 40 years old

Tenure: 3 years

Balance: \$ 60000

Number of Products: 2

Does this customer have a credit card ? Yes

Is this customer an Active Member: Yes

Estimated Salary: \$ 50000

So, should we say goodbye to that customer ?

**Solution**

The customer profile is encoded as:

| Feature | Value | Encoded |
|---------|-------|--------|
| Geography: France | | `1, 0, 0` |
| CreditScore | 600 | 600 |
| Gender: Male | | 1 |
| Age | 40 | 40 |
| Tenure | 3 | 3 |
| Balance | 60000 | 60000 |
| NumOfProducts | 2 | 2 |
| HasCrCard: Yes | | 1 |
| IsActiveMember: Yes | | 1 |
| EstimatedSalary | 50000 | 50000 |

In [ ]:
print(ann.predict(sc.transform([[1, 0, 0, 600, 1, 40, 3, 60000, 2, 1, 1, 50000]])) > 0.5)

The model predicts this customer will **stay** (probability < 0.5).

The raw output is a probability — `ann.predict(...)` returns a number like 0.12, meaning 12% chance of churning. Applying `> 0.5` converts this to a boolean decision.

**Tuning the threshold:** 0.5 is not always the right cutoff. If the cost of missing a churner (false negative) is much higher than a wasted retention offer (false positive), you might lower the threshold to 0.3 — flagging more customers as at-risk and accepting more false alarms to catch more real churners. This is a business decision, not a model decision.

### Step 13: Evaluate on the Full Test Set

The model predicts a probability for each of the 2,000 test customers. We convert to binary predictions using a 0.5 threshold.

The output shows `[predicted, actual]` pairs. Scanning them gives a qualitative picture before we compute the formal metrics.

In [ ]:
y_pred = ann.predict(X_test)
y_pred = (y_pred > 0.5)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

### Step 14: Confusion Matrix and Final Accuracy

The confusion matrix breaks down predictions into four categories:

```
                 Predicted Stay    Predicted Churn
Actual Stay         TN                  FP
Actual Churn        FN                  TP
```

**Reading the results (~86% accuracy):**

86% looks strong, but check the breakdown — churn prediction datasets are typically imbalanced (most customers stay). A naive model that predicts everyone stays would get ~80% accuracy too.

Look at the **false negatives (FN)** — churners the model missed. In a retention campaign, these are the most expensive errors. A high FN count means the model is not actually useful for the business goal, even at 86% accuracy.

**Better metrics for imbalanced churn:**

| Metric | What it tells you |
|--------|------------------|
| **Recall (Sensitivity)** | Of all churners, what fraction did we catch? |
| **Precision** | Of customers we flagged, how many actually churned? |
| **AUC-ROC** | Overall discrimination across all thresholds |
| **F1 Score** | Harmonic mean of precision and recall |

Always report these alongside accuracy for classification problems with class imbalance.

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)